In [ ]:
"""Q1.(1)코드"""
#(a)
import csv, json, logging
logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")
def make_report(csv_path: str, json_path: str) -> int: # 콜론 추가
    students = []
    try:
        with open(csv_path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                name = row['이름']
                student_id = row['학번']
                try:
                    mid = int(row['중간'])
                    fin = int(row['기말'])
                    asg = int(row['과제'])
                    #(b)
                    avg = mid * 0.3 + fin * 0.5 + asg * 0.2
                    if avg >= 90: 
                        grade = "A"
                    elif avg >= 80:
                        grade = "B"
                    elif avg >= 70:
                        grade = "C"
                    else:
                        grade = "F"
                    
                    logging.info(f"{name}: 평균 {avg:.1f}, 등급 {grade}") 
                except (ValueError):
                    avg = None
                    grade = None
                    logging.info(f"{name}: 성적에 결측값이 존재")
                
                students.append({
                    "이름": name,
                    "학번": student_id,
                    "점수": {"중간": int(row['중간']) if row['중간'] else None, 
                    "기말": int(row['기말']) if row['기말'] else None, 
                    "과제": int(row['과제']) if row['과제'] else None
                    },
                    "평균": round(avg, 1) if avg is not None else None,
                    "등급": grade
                })
        #(c)
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(students, f, ensure_ascii=False, indent=2)          
        return len(students)
    #(d)    
    except FileNotFoundError: 
        logging.warning(f"파일이 없습니다: {csv_path}")
        return 0     
    except UnicodeDecodeError:
        logging.error(f"인코딩 오류가 발생했습니다: {csv_path}")
        return 0

(2)

In [32]:
make_report("scores.csv", "out1.json")

[INFO] 김언어: 평균 89.5, 등급 B
[INFO] 이국문: 평균 84.4, 등급 B
[INFO] 박영문: 평균 93.5, 등급 A
[INFO] 최역사: 성적에 결측값이 존재


4

In [34]:
make_report("missing.csv", "out2.json")

[WARNING] 파일이 없습니다: missing.csv


0

In [33]:
make_report("euc-kr.csv", "out3.json")

[ERROR] 인코딩 오류가 발생했습니다: euc-kr.csv


0

(3)
with 문은 정상 흐름과 예외 흐름 모두에서 파일을 자동으로 닫아 준다. csv를 dict 형태로 바꿨다. 점수 결측치 발생 시 ValueError 예외 처리를 통해 평균과 등급에 None을 부여해서 프로그램 중단 없이 계속 이어지게 하였다. 한글 텍스트의 표준인 utf-8을 명시했다. 또한 FileNotFoundError와 UnicodeDecodeError 발생 시 각각 WARNING과 ERROR 로그를 남기고 0을 반환하도록 설계하여 디버깅이 쉽도록 하였다.

In [ ]:
"""Q2.(1) 코드, (2) 실행결과"""
#(a)
class InvalidJamoError(ValueError):
    """한글 자모가 아닌 경우"""
#(b)
def classify_jamo(c: str) -> str:
    if not isinstance(c, str):
        raise TypeError(f"입력값은 문자열이어야 합니다: {c}")
    if len(c) != 1:
        raise ValueError(f"입력값은 한 글자여야 합니다: {c}")
    if 12593<=ord(c)<=12622:
        return "자음"
    elif 12623<=ord(c)<=12643:
        return "모음"
    else:
        raise InvalidJamoError(f"한글 자모가 아닙니다: {c}")
#(c)
inputs = ["ㄱ", "ㅏ", "ㄲ", "가", "AB", 5, "ㅎ", "ㅣ", ""]
answer = []
for i in inputs:
    try:
        j = classify_jamo(i)
        answer.append(j)
        print(j)
    except TypeError as e:
        print(f"[TypeError] {e}")
        answer.append(None)
    except InvalidJamoError as e:
        print(f"[InvalidJamoError] {e}")
        answer.append(None)
    except ValueError as e:
        print(f"[ValueError] {e}")
        answer.append(None)
print("답:", answer)

자음
모음
자음
[InvalidJamoError] 한글 자모가 아닙니다: 가
[ValueError] 입력값은 한 글자여야 합니다: AB
[TypeError] 입력값은 문자열이어야 합니다: 5
자음
모음
[ValueError] 입력값은 한 글자여야 합니다: 
답: ['자음', '모음', '자음', None, None, None, '자음', '모음', None]


(3)InvalidJamoError은 입력값의 타입은 맞으나 그 값이 유효한 자모 범위가 아닐 때 발생하는 오류이기 때문에 ValueError에 포함된다. 가능하면 구체적인 예외 종류를 명시해야하므로 Exception보다는 ValueError의 자식으로 정의하는 것이 옳다.  함수 구현 할 때 TypeError와 ValueError를 통해 타입과 길이를 판별하고, 자모 판별 시에는 InvalidJamoError를 사용하여 대응하였다. 그리고 실제 리스트로 답을 구할 때 InvalidJamoError가 ValueError의 자식이므로 ValueError를 먼저 코드로 적을 시 InvalidJamoError에 해당하는 것들도 그 코드에 포함이 되므로 except InvalidJamoError가 except ValueError보다 먼저 왔다.